In [1]:
using SpecialFunctions
using LinearAlgebra
using Plots, LaTeXStrings
using Struve
using KrylovKit
using SparseArrays
using Arpack  # For sparse eigenvalue computations
using QuadGK
using ForwardDiff
using FFTW
using Statistics
using Measures
using Base.Threads
using TensorOperations
include("module_full_vs_wannier.jl")
using .exciton

In [2]:
const FD2_STEPS = [-1,0.0, 1]
const FD2_COEFFS = [-1.0/2.0,0.0, 1.0/2.0]

3-element Vector{Float64}:
 -0.5
  0.0
  0.5

In [3]:
function bloch_shift(wannier; lattice, Nsample, deltak = 0.2)
	delta_kappa = deltak * lattice.b2 / Nsample # This shifts Bloch function

	newbloch = zeros(eltype(wannier), size(wannier))
	for xv ∈ 1:Nsample, yv ∈ 1:Nsample
		k = (xv * lattice.b1 + yv * lattice.b2) / Nsample

		for xi ∈ 1:Nsample, yi ∈ 1:Nsample
			xi_reduced = periodic_cut_discrete(xi, Nsample, 20)
			yi_reduced = periodic_cut_discrete(yi, Nsample, 20)
			newbloch[xv, yv, :] += wannier[xi, yi, :] * exp(-im * k' * (xi * lattice.R1 + yi * lattice.R2)) * exp(-im * delta_kappa' * (xi_reduced * lattice.R1 + yi_reduced * lattice.R2))
		end
	end

	return newbloch
end

function periodic_cut_discrete(x::Integer, L::Integer, cutoff)
	halfL = div(L, 2)  # integer division
	# Wrap x into [-halfL, L - halfL - 1] for odd L
	y = mod1(x, L)
	if y > halfL
		y -= L
	end
	return abs(y) <= cutoff ? y : 0
end

function TMD_jx_6th(k; epsilonyy, sz, deltax = 0.1)
	kx, ky = k
	h = deltax
	f(x) = exciton.TMD_hnn((x, ky); epsilonyy, sz)

	return -(f(kx - 3h) - 9f(kx - 2h) + 45f(kx - h) - 45f(kx + h) + 9f(kx + 2h) - f(kx + 3h)) / (60h)
end

function TMD_jy_6th(k; epsilonyy, sz, deltay = 0.1)
	kx, ky = k
	h = deltay
	f(y) = exciton.TMD_hnn((kx, y); epsilonyy, sz)

	return -(f(ky - 3h) - 9f(ky - 2h) + 45f(ky - h) - 45f(ky + h) + 9f(ky + 2h) - f(ky + 3h)) / (60h)
end

function bse_kernel_construction(newbv, newbc1, newbc2, Nsample; VInt, epsilonyy, sz = 1, deltak, lattice)
	V = VInt.V
	# Precompute all Hamiltonian indices
	ham_indices = Array{Int}(undef, Nsample, Nsample, 2)
	for kx in 1:Nsample, ky in 1:Nsample, k_cond_ind in 1:2
		ham_indices[kx, ky, k_cond_ind] = ham_index(kx, ky, k_cond_ind; xlength = Nsample, ylength = Nsample)
	end

	ham = zeros(ComplexF64, Nsample^2 * 2, Nsample^2 * 2)
	conductions = [newbc1, newbc2]

	# Coulomb part
	for kx in 1:Nsample, ky in 1:Nsample, k_cond_ind in 1:2
		kidx = ham_indices[kx, ky, k_cond_ind]
		for kxp in 1:Nsample, kyp in 1:Nsample, kp_cond_ind in 1:2
			kpidx = ham_indices[kxp, kyp, kp_cond_ind]

			ck = conductions[k_cond_ind][kx, ky, :]
			ckp = conductions[kp_cond_ind][kxp, kyp, :]

			vk = newbv[kx, ky, :]
			vkp = newbv[kxp, kyp, :]
			qx, qy = mod1.([kxp - kx, kyp - ky], Nsample)
			ham[kidx, kpidx] += -(ck' * ckp) * (vkp' * vk) * V[qx, qy] / Nsample^2 #
		end
	end

	# Add ecvmat contribution (same k)
	for kx in 1:Nsample, ky in 1:Nsample
		kvec = (kx * lattice.b1 + (ky+deltak) * lattice.b2) / Nsample
		hkmat = exciton.TMD_hnn(kvec; epsilonyy, sz)
		c1 = newbc1[kx, ky, :]
		c2 = newbc2[kx, ky, :]
		v = newbv[kx, ky, :]
		ev = v' * hkmat * v

		ecvmat = [c1'*hkmat*c1-ev c1'*hkmat*c2; c2'*hkmat*c1 c2'*hkmat*c2-ev]
		for k_cond_ind in 1:2, kp_cond_ind in 1:2
			kidx = ham_indices[kx, ky, k_cond_ind]
			kpidx = ham_indices[kx, ky, kp_cond_ind]
			ham[kidx, kpidx] += ecvmat[k_cond_ind, kp_cond_ind]
		end
	end

	return ham

end


"""
Helper: Shifts a wavefunction by multiple steps of dk.
"""
function generate_shifted_bloch(w, lattice, Nsample, dk, steps)
	return [bloch_shift(w; lattice, Nsample, deltak = s * dk) for s in steps]
end

"""
Helper: Solves the BSE for a specific k-shift to get the Envelope Function (Psi).
"""
function solve_bse_at_shift(bv, bc1, bc2, Nsample, VInt, epsilonyy, shift_idx, dk, state_idx; lattice)
	# Calculate actual deltak for the kernel construction
	current_deltak = FD2_STEPS[shift_idx] * dk

	bsemat = bse_kernel_construction(bv, bc1, bc2, Nsample;
		VInt, epsilonyy, sz = 1, deltak = current_deltak, lattice)

	xlen = size(bsemat)[1]
	x0 = rand(xlen)

	# Solve Eigenproblem
	valslist, vecslist, _ = eigsolve(bsemat, x0, 20, :SR,
		krylovdim = 120, tol = 1e-10, maxiter = 40,
		verbosity = 0, ishermitian = true)

	# Extract specific state and energy
	return vecslist[state_idx], valslist[1]
end

"""
Helper: Enforces phase continuity (U(1) gauge fixing) across the list of Psis.
"""
function fix_gauge!(psi_list)
	for n in 1:(length(psi_list)-1)
		overlap = dot(psi_list[n], psi_list[n+1])
		if abs(overlap) > 1e-12
			phase = overlap / abs(overlap) # e^{iθ}
			# Rotate n+1 to match n
			psi_list[n+1] .*= conj(phase)
		end
	end
end

"""
Term 1: Derivative of the Optical Matrix Element Phase (Wilson Line).
"""
function compute_optical_phase_deriv(bv_list, bc1_list, bc2_list, psi_list,
	lattice, Nsample, dk, polarization, epsilonyy, sz)

	phases = zeros(Float64, length(FD2_STEPS))

	for (i, step_mult) in enumerate(FD2_STEPS)
		pump_sum = ComplexF64(0.0)

		# Grid loop
		for xdim ∈ 1:Nsample, ydim ∈ 1:Nsample
			# Calculate k for current shift
			k = (xdim * lattice.b1 + (ydim + dk * step_mult) * lattice.b2) / Nsample

			# Dipole operators
			jx = TMD_jx_6th(k; epsilonyy, sz)
			jy = TMD_jy_6th(k; epsilonyy, sz)
			j_op = polarization[1] * jx + polarization[2] * jy

			# Indices
			p1_idx = ham_index(xdim, ydim, 1; xlength = Nsample, ylength = Nsample)
			p2_idx = ham_index(xdim, ydim, 2; xlength = Nsample, ylength = Nsample)

			# Wavefunctions at this grid point
			v = bv_list[i][xdim, ydim, :]
			c1 = bc1_list[i][xdim, ydim, :]
			c2 = bc2_list[i][xdim, ydim, :]

			# Optical transition elements weighted by Envelope Psi
			pump_sum += (v' * j_op * c1) * psi_list[i][p1_idx]
			pump_sum += (v' * j_op * c2) * psi_list[i][p2_idx]
		end
		phases[i] = angle(pump_sum)
	end

	# Apply 4th order finite difference to the scalar phases
	deriv = sum(phases .* FD2_COEFFS)
	return deriv
end

"""
Term 2: Berry Connection of the Bloch Bands projected onto Envelope.
Term 3: Derivative of the Envelope Function (Psi).
"""
function compute_berry_terms(bv_list, bc1_list, bc2_list, psi_list, Nsample)

	# Indices for the center point (k) within the lists
	center_idx = findfirst(x -> x == 0, FD2_STEPS) # Should be 3

	# Pre-fetch center wavefunctions
	v_center = bv_list[center_idx]
	c1_center = bc1_list[center_idx]
	c2_center = bc2_list[center_idx]
	psi_center = psi_list[center_idx]

	term2 = ComplexF64(0.0)

	# We need to compute <u(k) | d/dk | u(k)>.
	# d/dk u(k) approx sum(coeffs[j] * u(k+j*dk))

	for kx ∈ 1:Nsample, ky ∈ 1:Nsample
		# 1. Compute Finite Difference Derivatives of Bloch functions at k (center)
		# Initialize derivatives as zero vectors
		dv = zeros(ComplexF64, length(v_center[kx, ky, :]))
		dc1 = zeros(ComplexF64, length(c1_center[kx, ky, :]))
		dc2 = zeros(ComplexF64, length(c2_center[kx, ky, :]))

		for (j, coeff) in enumerate(FD2_COEFFS)
			dv .+= coeff .* bv_list[j][kx, ky, :]
			dc1 .+= coeff .* bc1_list[j][kx, ky, :]
			dc2 .+= coeff .* bc2_list[j][kx, ky, :]
		end

		# 2. Compute Berry Connections A = i <u | du> (un-normalized by dk here, handled in coeffs)
		# Note: These are scalar values for specific kx, ky
		Av = im * dot(v_center[kx, ky, :], dv)
		Ac11 = im * dot(c1_center[kx, ky, :], dc1)
		Ac12 = im * dot(c1_center[kx, ky, :], dc2)
		Ac21 = im * dot(c2_center[kx, ky, :], dc1)
		Ac22 = im * dot(c2_center[kx, ky, :], dc2)

		# 3. Project onto Envelope Function (Psi)
		p1 = ham_index(kx, ky, 1; xlength = Nsample, ylength = Nsample)
		p2 = ham_index(kx, ky, 2; xlength = Nsample, ylength = Nsample)

		psi1 = psi_center[p1]
		psi2 = psi_center[p2]

		term2 += (Ac11 - Av) * conj(psi1) * psi1
		term2 += (Ac12) * conj(psi1) * psi2
		term2 += (Ac21) * conj(psi2) * psi1
		term2 += (Ac22 - Av) * conj(psi2) * psi2
	end

	# Term 3: Envelope Derivative <Psi | d/dk | Psi>
	# d/dk Psi approx sum(coeffs[j] * Psi[j])
	dPsi = zeros(ComplexF64, length(psi_center))
	for (j, coeff) in enumerate(FD2_COEFFS)
		dPsi .+= coeff .* psi_list[j]
	end

	term3 = im * dot(psi_center, dPsi)

	return term2, term3
end

# --- Main Subroutine ---

function exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(pi / 3), sin(pi / 3)]], VInt, lattice, epsilonyy, sz = 1, Nsample)

	dk = 0.01

	# 1. Generate Shifted Bloch Functions (5 points: -2dk to +2dk)
	bv_list = generate_shifted_bloch(wv, lattice, Nsample, dk, FD2_STEPS)
	bc1_list = generate_shifted_bloch(wc1, lattice, Nsample, dk, FD2_STEPS)
	bc2_list = generate_shifted_bloch(wc2, lattice, Nsample, dk, FD2_STEPS)

	# 2. Compute Envelope Functions (Psi) for each shift
	psi_list = Vector{Vector{ComplexF64}}(undef, length(FD2_STEPS))

	#   println("Starting Diagonalization...")
	for i in 1:length(FD2_STEPS)
		psi, val = solve_bse_at_shift(bv_list[i], bc1_list[i], bc2_list[i],
			Nsample, VInt, epsilonyy, i, dk, state; lattice)
		psi_list[i] = psi
		if i == 2 # Center point
			println("Center Energy: ", val)
		end
	end
	#  println("Finish computing envelope functions")

	# 3. Gauge Fixing (Critical for derivatives)
	# We copy to preserve raw data if needed, though strictly not necessary if memory is abundant
	psi_list_gauge_fixed = deepcopy(psi_list)
	fix_gauge!(psi_list_gauge_fixed)

	# 4. Compute Derivatives
	# Normalization factor for derivative: 1 / (dk * |b2|/Nsample)
	dk_norm = (dk * norm(lattice.b2) / Nsample)

	# A. Optical Phase Derivative (Wilson-like term)
	# The coeffs are applied inside, so result is d(Angle)/d(Step)
	#   term1_val = compute_optical_phase_deriv(bv_list, bc1_list, bc2_list, psi_list_gauge_fixed,
	#       lattice, Nsample, dk, polarization, epsilonyy, sz)

	# B. Berry Connections and Envelope Derivative
	# These return raw sums weighted by coeffs. We must divide by dk_norm at the end.
	term2_val, term3_val = compute_berry_terms(bv_list, bc1_list, bc2_list, psi_list_gauge_fixed, Nsample)

	rtot_list = Vector{ComplexF64}(undef, length(polarization))

	for (j, pol) in pairs(polarization)

		# Term 1 for this polarization
		term1_val = compute_optical_phase_deriv(
			bv_list, bc1_list, bc2_list, psi_list_gauge_fixed,
			lattice, Nsample, dk, pol, epsilonyy, sz
		)

		# rtot(j)
		rtot_list[j] = (term1_val + term2_val + term3_val) / dk_norm
	end

	println("rtot for all polarizations: ", rtot_list)

	# Sum and Normalize
	# FD4_COEFFS assumes sum(c_i * f_i), so we divide by the step size (dk_norm) once.
	#   rtot = (term1_val + term2_val + term3_val) / dk_norm

	return rtot_list
end


println("Start calculation")
Nsample=60
sz = 1
lattice = GrapheneLattice();
#VInt = InteractionMatrix(lattice, Nsample; lambda=0.8, r0=4.28)
VInt = InteractionMatrix(lattice, Nsample; lambda = 0.667, r0 = 4.288)
bi_1 = lattice.b1 / Nsample
bi_2 = lattice.b2 / Nsample
bi_3 = -bi_1 - bi_2
wb = 2 / (3 * norm(bi_1)^2)

# --- CRITICAL PERFORMANCE SETTING ---
# We force linear algebra to be single-threaded so our 4 Julia threads
# can run on independent cores without fighting for resources.
BLAS.set_num_threads(1)

Start calculation


In [ ]:
strainlist = -6:0.5:6

# Pre-allocate results (Thread-safe for writing if indices are unique)
wannierlist = Vector{ComplexF64}(undef, length(strainlist))
blochlist = Vector{Vector{ComplexF64}}(undef, length(strainlist))

# Parallelize the outer loop
Threads.@threads for i in 1:length(strainlist)
	strain_val = strainlist[i]
	# --- Everything below is local to this specific thread/strain ---
	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	# 1. Initialize System
	# Note: 'lattice', 'Nsample', 'sz' must be defined in global scope
	graphene_sys = GrapheneBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)

	bloch_v, bloch_c1, bloch_c2 = graphene_sys.Bloch.valence, graphene_sys.Bloch.conduction1, graphene_sys.Bloch.conduction2
	bloch_states = [bloch_v]

	M_b1 = exciton.compute_M_matrix(bloch_states, (1, 0))
	M_b2 = exciton.compute_M_matrix(bloch_states, (0, 1))
	M_b3 = exciton.compute_M_matrix(bloch_states, (-1, -1))

	blist = [bi_1, bi_2, bi_3]
	M_blist = [M_b1, M_b2, M_b3]
	nbands = length(bloch_states)

	rlist = [
		sum(-wb / Nsample^2 * blist[bind] * angle(M_blist[bind][xi, yi, band, band])
			for xi ∈ 1:Nsample, yi ∈ 1:Nsample, bind ∈ 1:3)
		for band ∈ 1:nbands
	]

	# 2. First Descent (Valence)
	U0mn_mat = zeros(ComplexF64, Nsample, Nsample, nbands, nbands)
	for xi ∈ 1:Nsample, yi ∈ 1:Nsample, nband_ind ∈ 1:nbands
		U0mn_mat[xi, yi, nband_ind, nband_ind] = 1
	end

	updated_Umn_mat = exciton.one_descent_step_singleband(U0mn_mat; M0_blist = M_blist, wb, varied_step = false, Nsample, blist, rlist, verbose = false)
	for steps ∈ 1:200
		updated_Umn_mat = exciton.one_descent_step_singleband(updated_Umn_mat; M0_blist = M_blist, wb, varied_step = false, alphaval = 0.1, Nsample, blist, rlist, verbose = false)
	end

	oldv = graphene_sys.Bloch.valence
	newv = 0 * oldv
	for xi ∈ 1:Nsample, yi ∈ 1:Nsample
		newv[xi, yi, :] = updated_Umn_mat[xi, yi, 1, 1] * oldv[xi, yi, :]
	end

	# 3. Second Descent (Conduction)
	bloch_states = [bloch_c1, bloch_c2]
	M_b1 = exciton.compute_M_matrix(bloch_states, (1, 0))
	M_b2 = exciton.compute_M_matrix(bloch_states, (0, 1))
	M_b3 = exciton.compute_M_matrix(bloch_states, (-1, -1))

	M_blist = [M_b1, M_b2, M_b3]
	nbands = length(bloch_states)

	rlist = [
		sum(-wb / Nsample^2 * blist[bind] * angle(M_blist[bind][xi, yi, band, band])
			for xi ∈ 1:Nsample, yi ∈ 1:Nsample, bind ∈ 1:3)
		for band ∈ 1:nbands
	]

	U0mn_mat = zeros(ComplexF64, Nsample, Nsample, nbands, nbands)
	for xi ∈ 1:Nsample, yi ∈ 1:Nsample, nband_ind ∈ 1:nbands
		U0mn_mat[xi, yi, nband_ind, nband_ind] = 1
	end

	updated_Umn_mat = exciton.one_descent_step(U0mn_mat; M0_blist = M_blist, wb, nbands, Nsample, blist, rlist, verbose = false)
	for steps ∈ 1:200
		updated_Umn_mat = exciton.one_descent_step(updated_Umn_mat; M0_blist = M_blist, wb, nbands, Nsample, blist, rlist, verbose = false)
	end

	oldc1 = graphene_sys.Bloch.conduction1
	oldc2 = graphene_sys.Bloch.conduction2
	newc1 = 0 * oldc1
	newc2 = 0 * oldc2

	for xi ∈ 1:Nsample, yi ∈ 1:Nsample
		newc1[xi, yi, :] = updated_Umn_mat[xi, yi, 1, 1] * oldc1[xi, yi, :] + updated_Umn_mat[xi, yi, 2, 1] * oldc2[xi, yi, :]
		newc2[xi, yi, :] = updated_Umn_mat[xi, yi, 1, 2] * oldc1[xi, yi, :] + updated_Umn_mat[xi, yi, 2, 2] * oldc2[xi, yi, :]
	end

	oldecv = graphene_sys.Bloch.ecvmat
	newecv = 0 * oldecv
	for xi ∈ 1:Nsample, yi ∈ 1:Nsample
		urotated = updated_Umn_mat[xi, yi, :, :]
		newecv[xi, yi, :, :] = urotated' * oldecv[xi, yi, :, :] * urotated
	end

	# 4. Construct MLWF System and Solve BSE
	mlwf_bloch = BlochStates(newv, newc1, newc2, newecv)

	# NOTE: We shadow the variable 'graphene_sys' locally so it doesn't conflict
	local_sys = GrapheneBSE(graphene_sys.lattice, graphene_sys.kappa, graphene_sys.Nsample, mlwf_bloch, nothing, nothing)
	local_sys = add_bse_kernel(local_sys, VInt)

	bsemat = local_sys.BSEKernel
	xlen = size(bsemat)[1]
	x0 = rand(xlen) # Random starting vector

	valslist, vecslist, info = eigsolve(bsemat, x0, 10, :SR,
		krylovdim = 40, tol = 1e-10, maxiter = 20,
		verbosity = 0, ishermitian = true)

	local_sys = add_wannier(local_sys)
	psik = vecslist[1]
	psi_real = exciton.envelope_real_fft(psik, local_sys)
	r1 = Rshift_calculator(local_sys, psi_real, div(Nsample, 2) - 2)

	# 5. Save Results Safely
	# Since 'i' is unique for each thread, this write is safe.
	wannierlist[i] = r1[2]
	flush(stdout)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	blochlist[i] = exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(2pi / 3), sin(2pi / 3)], [cos(3pi / 4), sin(3pi / 4)], [cos(5pi / 6), sin(5pi / 6)]], VInt, lattice, epsilonyy, sz = 1, Nsample)
end

polarization_list=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list[i]=real(blochlist[i][1])
end

polarization_list2=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list2[i]=real(blochlist[i][2])
end

polarization_list3=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list3[i]=real(blochlist[i][3])
end

sigma1=polarization_list .- real.(wannierlist)
println("std1 is ", sqrt(sum(sigma1 .^ 2 ./ 25)))
sigma2=polarization_list2 .- real.(wannierlist)
println("std2 is ", sqrt(sum(sigma2 .^ 2 ./ 25)))
sigma3=polarization_list3 .- real.(wannierlist)
println("std3 is ", sqrt(sum(sigma3 .^ 2 ./ 25)))

hline([0], color = :black, lw = 0.5, label = false, xlim = (-6.1, 6.1), ylim = (-0.4, 0.4), xticks = ([-6, -3, 0, 3, 6]))

# Vertical axis at x = 0
vline!([0], color = :black, lw = 0.5, label = false)
scatter!(
	strainlist,
	[real.(wannierlist), polarization_list, polarization_list2, polarization_list3] .* 100,
	framestyle = :box,
	xgrid = false,
	ygrid = false,
	label = ["Wannier" "Full, "*L"\theta=2\pi/3" "Full, "*L" \theta=3\pi/4" "Full, "*L" \theta=5\pi/6"],
	tickfontsize = 12,
	dpi = 800,
	size = (400, 280),
	markershape = [:circle :square :utriangle :diamond],
	ms = [5.5 4 4 5],
	markeralpha = 0.6,
)

savefig("figs2_multithread.png")

Thread 4 doing strain 0.5 (Index 14)
Thread 2 doing strain 3.5 (Index 20)
Thread 1 doing strain -6.0 (Index 1)
Thread 3 doing strain -2.5 (Index 8)
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
